# 03 Experiment Semantic Layer


In [1]:
# ============================================================
# 03_experiment_semantic_layer.ipynb
# Semantic Layer 실험:
#   추상화된 비즈니스 개념 → LLM → SQL 생성 → 실행 정확도 측정
#   + 민감정보 노출 0개 검증
# ============================================================

# %% [1] 라이브러리 & 환경 설정
import os
import json
import time
import sqlite3
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from tqdm import tqdm

# ── 경로
ROOT    = Path(os.getcwd())
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DB_PATH  = ROOT / "data"      / "insurance_uw.db"
Q1_PATH  = ROOT / "benchmark" / "questionset_v2_part1.json"
Q2_PATH  = ROOT / "benchmark" / "questionset_v2_part2.json"
SL_DIR   = ROOT / "semantic_layer"
RAW_DIR  = ROOT / "results"   / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
SL_DIR.mkdir(parents=True, exist_ok=True)

# ── API 키
load_dotenv(ROOT / ".env")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
LLM_MODEL      = os.getenv("LLM_MODEL", "gpt-4o-mini")

if not OPENAI_API_KEY:
    raise ValueError("[ERROR] OPENAI_API_KEY 없음 → .env 확인")

client = OpenAI(api_key=OPENAI_API_KEY)

# ── 실험 설정
N_REPEAT  = 3
DELAY_SEC = 0.3
TIMEOUT_S = 25

print("=" * 55)
print("  03_experiment_semantic_layer")
print("=" * 55)
print(f"Model    : {LLM_MODEL}")
print(f"반복 횟수 : {N_REPEAT}회")
assert DB_PATH.exists(), "DB 없음 → 00_setup 먼저 실행"


# %% [2] 질문셋 로드
def load_qs(path: Path) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        d = json.load(f)
    return d if isinstance(d, list) else d.get("questions", [])

all_qs = load_qs(Q1_PATH) + load_qs(Q2_PATH)
print(f"\n질문셋 로드: {len(all_qs)}개")


# %% [3] Semantic Layer 정의
# ── 핵심 원칙 (Parnas 1972 Information Hiding):
#    LLM에게 원시 컬럼명(insured_name, diagnosis_code 등) 대신
#    비즈니스 개념(피보험자, 진단코드)만 노출
#    실제 컬럼 매핑은 SQL_MAPPING에서 처리

SEMANTIC_LAYER = {
    "cubes": [
        {
            "name": "계약",
            "description": "보험계약 정보. 보험료, 보험금, 계약 상태, 심사 결과 등 포함.",
            "dimensions": [
                {"name": "계약ID",        "description": "계약 고유 식별자"},
                {"name": "상품유형",       "description": "상품 분류 (protection/savings/annuity/third_insurance/variable)"},
                {"name": "계약상태",       "description": "계약 현황 (active/terminated/expired/withdrawn)"},
                {"name": "해지사유",       "description": "계약 해지 이유 (고지의무위반/중대사유/납입연체 등)"},
                {"name": "취소사유",       "description": "청약 취소 이유"},
                {"name": "계약일",         "description": "청약일 (ISO 8601)"},
                {"name": "보장개시일",     "description": "보험 보장이 시작되는 날짜"},
                {"name": "해지일",         "description": "계약 해지 날짜"},
                {"name": "연금개시일",     "description": "연금 지급 시작 날짜"},
                {"name": "계약유형",       "description": "진단계약 또는 무진단계약"},
                {"name": "보장유형",       "description": "사망/장해/입원/연금 등"},
                {"name": "고지의무위반입증여부", "description": "위반 사실이 보험금 지급에 영향을 미쳤는지 (1=입증됨)"},
                {"name": "사기유형",       "description": "사기 유형 (고의보험사고유발/서류위변조 등)"},
                {"name": "나이오류수정여부", "description": "보험나이 착오로 보험료가 정정된 경우 (1=수정됨)"},
                {"name": "통화",           "description": "보험료 통화 (KRW/USD 등)"},
                {"name": "장해지급률재결정여부", "description": "장해 악화로 지급률이 재결정된 경우 (1=재결정)"},
                {"name": "건강진단완료여부", "description": "진단계약에서 건강진단 완료 여부 (1=완료)"},
                {"name": "타인사망보장여부", "description": "타인 사망을 보험금 지급사유로 하는지 (1=해당)"},
                {"name": "피보험자서면동의여부", "description": "피보험자의 서면 동의 여부 (1=동의)"},
            ],
            "measures": [
                {"name": "월보험료",       "description": "월 납입 보험료 (원)"},
                {"name": "연보험료",       "description": "연 납입 보험료 (원)"},
                {"name": "기납입보험료",   "description": "지금까지 납입한 총 보험료 (원)"},
                {"name": "보험가입금액",   "description": "가입한 보험금 총액 (원)"},
                {"name": "사망보험금액",   "description": "사망 시 지급되는 보험금 (원)"},
                {"name": "해지환급금",     "description": "계약 해지 시 돌려받는 금액 (원)"},
                {"name": "계약자적립액",   "description": "현재 적립된 계약자 몫의 금액 (원)"},
                {"name": "대출잔액",       "description": "보험계약대출 잔액 (원)"},
                {"name": "납입횟수",       "description": "현재까지 납입 완료한 횟수"},
                {"name": "납입경과연수",   "description": "납입 시작 후 경과한 연수"},
                {"name": "계약시보험나이", "description": "계약 체결 시점의 보험나이 (세)"},
                {"name": "수정전보험료",   "description": "나이 착오 수정 전 보험료 (원)"},
                {"name": "수정후보험료",   "description": "나이 착오 수정 후 보험료 (원)"},
                {"name": "현재원화환산보험료", "description": "외화보험의 현재 원화 환산 월보험료"},
                {"name": "계약수",         "description": "계약 건수 집계"},
            ],
        },
        {
            "name": "청구",
            "description": "보험금 청구 및 지급 정보.",
            "dimensions": [
                {"name": "청구유형",       "description": "청구 종류 (사망보험금/장해보험금/입원보험금/만기보험금)"},
                {"name": "청구상태",       "description": "처리 상태 (지급완료/지급거절/심사중/소멸시효완성)"},
                {"name": "지급방식",       "description": "일시지급 또는 분할지급"},
                {"name": "지급거절사유",   "description": "지급 거절 이유 (수익자고의피해/재해분류제외 등)"},
                {"name": "사망원인",       "description": "사망 원인 (자살/질병/재해 등)"},
                {"name": "사망처리유형",   "description": "실종선고 등 특수 사망 처리 유형"},
                {"name": "장해유형",       "description": "장해 종류 (심한추간판탈출증/흉복부장기심한장해 등)"},
                {"name": "장해신체부위",   "description": "장해 발생 신체 부위 (눈/귀/척추/팔/다리 등)"},
                {"name": "치매CDR점수",   "description": "임상치매척도 점수 (2~5점)"},
                {"name": "주진단명",       "description": "주요 진단 (뇌졸중/뇌손상 등)"},
                {"name": "조사동의여부",   "description": "보험금 지급사유 조사 동의 여부 (1=동의)"},
                {"name": "가지급여부",     "description": "가지급보험금 지급 여부 (1=지급)"},
                {"name": "청구일",         "description": "보험금 청구 접수일"},
                {"name": "지급예정일",     "description": "보험금 지급 예정일"},
                {"name": "실지급일",       "description": "실제 보험금 지급일"},
                {"name": "장해평가일",     "description": "장해 정도를 판정한 날짜"},
                {"name": "발병일",         "description": "질병 또는 재해 발병일"},
            ],
            "measures": [
                {"name": "지급보험금",     "description": "실제 지급된 보험금액 (원)"},
                {"name": "가산이자",       "description": "지연 지급으로 발생한 가산이율 금액 (원)"},
                {"name": "가지급금액",     "description": "가지급보험금 금액 (원)"},
                {"name": "영업일수",       "description": "청구 접수부터 지급까지 영업일 수"},
                {"name": "장해지급률",     "description": "장해 분류표에 따른 지급률 (%)"},
                {"name": "장해확정일수",   "description": "재해일/진단일부터 장해지급률 확정까지 일수"},
                {"name": "실종일수",       "description": "실종선고의 실종 기간 (일)"},
                {"name": "추가이자",       "description": "분할지급 시 평균공시이율로 가산된 이자 (원)"},
                {"name": "청구수",         "description": "청구 건수 집계"},
            ],
        },
        {
            "name": "피보험자",
            "description": "피보험자 인적사항.",
            "dimensions": [
                {"name": "피보험자ID",     "description": "피보험자 고유 식별자"},
                {"name": "성별",           "description": "성별 (M/F)"},
            ],
            "measures": [
                {"name": "가입연령",       "description": "보험 가입 시점의 나이 (세)"},
                {"name": "현재보험나이",   "description": "현재 보험나이 (세)"},
                {"name": "피보험자수",     "description": "피보험자 수 집계"},
            ],
        },
        {
            "name": "상품",
            "description": "보험상품 정보.",
            "dimensions": [
                {"name": "상품명",         "description": "상품 이름"},
                {"name": "상품분류",       "description": "상품 대분류 (protection/savings/annuity 등)"},
                {"name": "보험기간",       "description": "보장 기간 (whole_life/term/숫자)"},
                {"name": "보험기간년수",   "description": "보험기간 (년)"},
                {"name": "납입기간년수",   "description": "보험료 납입 기간 (년)"},
                {"name": "자동갱신여부",   "description": "자동갱신 상품 여부 (1=해당)"},
                {"name": "무해약환급금형여부", "description": "무해약환급금형 상품 여부 (1=해당)"},
                {"name": "연금후사망보험금여부", "description": "연금 개시 후에도 사망보험금 지급 여부 (1=해당)"},
                {"name": "보험유형",       "description": "생명보험/비생명보험 구분"},
                {"name": "실손보험소분류", "description": "실손의료보험 세분류 (basic/elderly/high_risk)"},
                {"name": "공시이율적용여부", "description": "금리연동형 여부"},
            ],
            "measures": [
                {"name": "생존보험금합계", "description": "생존 시 지급 예정 보험금 총합 (원)"},
                {"name": "예상납입보험료", "description": "전체 납입 예상 보험료 (원)"},
                {"name": "최저보증이율",   "description": "금리연동형 최저 보증이율 (%)"},
                {"name": "입원공제율",     "description": "실손의료보험 입원 시 공제 비율 (소수점)"},
                {"name": "계약체결비용",   "description": "계약 체결에 사용되는 비용 (원)"},
                {"name": "표준해약공제액", "description": "표준형 해약 공제금액 (원)"},
                {"name": "초기비용집중비율", "description": "납입기간 대비 초기 집중 계약체결비용 비율"},
            ],
        },
        {
            "name": "언더라이팅",
            "description": "보험 인수 심사 결과.",
            "dimensions": [
                {"name": "심사결과",       "description": "승낙/조건부승낙/거절"},
                {"name": "조건유형",       "description": "조건부 승낙 조건 (보험료할증/보장제외/보험금삭감/보험가입금액제한)"},
                {"name": "심사일",         "description": "심사 결정 날짜"},
            ],
            "measures": [
                {"name": "보험료할증률",   "description": "표준 대비 할증 비율 (소수점, 예: 0.2 = 20% 할증)"},
                {"name": "위험률할증률",   "description": "위험률 기준 할증 비율"},
                {"name": "추가할증률",     "description": "제3보험 15년 초과 추가 할증 비율"},
                {"name": "할증전위험보험료", "description": "할증 적용 전 위험보험료 (원)"},
                {"name": "심사건수",       "description": "심사 건수 집계"},
            ],
        },
        {
            "name": "모집",
            "description": "보험 모집 및 판매 채널 정보.",
            "dimensions": [
                {"name": "모집채널",       "description": "판매 경로 (telemarketing/bancassurance/agency/direct)"},
                {"name": "음성녹음완료여부", "description": "통신판매 음성녹음 완료 여부 (1=완료)"},
                {"name": "품질점검결과",   "description": "표준설명대본 점검 결과 (pass/insufficient_explanation/fail)"},
                {"name": "점검일",         "description": "품질 점검 날짜"},
            ],
            "measures": [
                {"name": "모집건수",       "description": "모집 건수 집계"},
            ],
        },
    ]
}

# YAML 저장
import yaml
sl_yaml_path = SL_DIR / "schema_context.yaml"
with open(sl_yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(SEMANTIC_LAYER, f, allow_unicode=True, sort_keys=False)
print(f"Semantic Layer 정의 저장: {sl_yaml_path.name}")


# %% [4] SQL 매핑 정의 (개념 → 실제 SQL 표현식)
SQL_MAPPING = {
    # ── 계약 차원
    "계약ID"              : "contracts.contract_id",
    "상품유형"            : "products.product_category",
    "계약상태"            : "contracts.contract_status",
    "해지사유"            : "contracts.termination_reason",
    "취소사유"            : "contracts.cancellation_reason",
    "계약일"              : "contracts.contract_date",
    "보장개시일"          : "contracts.coverage_start_date",
    "해지일"              : "contracts.termination_date",
    "연금개시일"          : "contracts.annuity_start_date",
    "계약유형"            : "contracts.contract_type",
    "보장유형"            : "contracts.coverage_type",
    "고지의무위반입증여부": "contracts.causation_proven",
    "사기유형"            : "contracts.fraud_type",
    "나이오류수정여부"    : "contracts.age_correction_applied",
    "통화"                : "contracts.currency_code",
    "장해지급률재결정여부": "contracts.disability_rate_revised",
    "건강진단완료여부"    : "contracts.health_exam_completed",
    "타인사망보장여부"    : "contracts.third_party_death_coverage",
    "피보험자서면동의여부": "contracts.insured_written_consent",
    # ── 계약 측정값
    "월보험료"            : "contracts.monthly_premium",
    "연보험료"            : "contracts.annual_premium",
    "기납입보험료"        : "contracts.total_paid_premium",
    "보험가입금액"        : "contracts.sum_insured",
    "사망보험금액"        : "contracts.death_benefit",
    "해지환급금"          : "contracts.surrender_value",
    "계약자적립액"        : "contracts.policyholder_reserve",
    "대출잔액"            : "contracts.loan_balance",
    "납입횟수"            : "contracts.payment_count",
    "납입경과연수"        : "contracts.payment_year_elapsed",
    "계약시보험나이"      : "contracts.insured_age_at_contract",
    "수정전보험료"        : "contracts.original_premium",
    "수정후보험료"        : "contracts.corrected_premium",
    "현재원화환산보험료"  : "contracts.current_monthly_premium_krw",
    "계약수"              : "COUNT(contracts.contract_id)",
    # ── 청구 차원
    "청구유형"            : "claims.claim_type",
    "청구상태"            : "claims.claim_status",
    "지급방식"            : "claims.payment_method",
    "지급거절사유"        : "claims.claim_denial_reason",
    "사망원인"            : "claims.death_cause",
    "사망처리유형"        : "claims.death_cause_type",
    "장해유형"            : "claims.disability_type",
    "장해신체부위"        : "claims.disability_body_part",
    "치매CDR점수"         : "claims.cdr_score",
    "주진단명"            : "claims.primary_diagnosis",
    "조사동의여부"        : "claims.investigation_consent",
    "가지급여부"          : "claims.provisional_payment_made",
    "청구일"              : "claims.claim_date",
    "지급예정일"          : "claims.due_payment_date",
    "실지급일"            : "claims.actual_payment_date",
    "장해평가일"          : "claims.disability_assessment_date",
    "발병일"              : "claims.onset_date",
    # ── 청구 측정값
    "지급보험금"          : "claims.paid_amount",
    "가산이자"            : "claims.additional_interest_amount",
    "가지급금액"          : "claims.provisional_payment_amount",
    "영업일수"            : "claims.business_days_to_payment",
    "장해지급률"          : "claims.disability_payment_rate",
    "장해확정일수"        : "claims.disability_rate_fixed_days",
    "실종일수"            : "claims.missing_period_days",
    "추가이자"            : "claims.interest_added_amount",
    "청구수"              : "COUNT(claims.claim_id)",
    # ── 피보험자
    "피보험자ID"          : "insured.insured_id",
    "성별"                : "insured.gender",
    "가입연령"            : "insured.age_at_entry",
    "현재보험나이"        : "contracts.insurance_age",
    "피보험자수"          : "COUNT(DISTINCT insured.insured_id)",
    # ── 상품
    "상품명"              : "products.product_name",
    "상품분류"            : "products.product_category",
    "보험기간"            : "products.insurance_period",
    "보험기간년수"        : "products.insurance_period_years",
    "납입기간년수"        : "products.payment_period_years",
    "자동갱신여부"        : "products.auto_renewal_flag",
    "무해약환급금형여부"  : "products.low_surrender_value_type",
    "연금후사망보험금여부": "products.post_annuity_death_benefit_flag",
    "보험유형"            : "products.insurance_type",
    "실손보험소분류"      : "products.medical_insurance_subtype",
    "생존보험금합계"      : "products.survival_benefit_total",
    "예상납입보험료"      : "products.expected_total_premium",
    "최저보증이율"        : "products.min_guaranteed_rate",
    "입원공제율"          : "products.inpatient_deductible_rate",
    "계약체결비용"        : "products.contract_conclusion_cost",
    "표준해약공제액"      : "products.standard_surrender_charge",
    "초기비용집중비율"    : "products.front_loaded_cost_ratio",
    # ── 언더라이팅
    "심사결과"            : "underwriting_decisions.decision_type",
    "조건유형"            : "underwriting_decisions.condition_type",
    "심사일"              : "underwriting_decisions.decision_date",
    "보험료할증률"        : "underwriting_decisions.premium_surcharge_rate",
    "위험률할증률"        : "underwriting_decisions.risk_surcharge_rate",
    "추가할증률"          : "underwriting_decisions.extra_surcharge_rate",
    "할증전위험보험료"    : "underwriting_decisions.risk_premium_before_surcharge",
    "심사건수"            : "COUNT(underwriting_decisions.decision_id)",
    # ── 모집
    "모집채널"            : "sales_channels.channel_type",
    "음성녹음완료여부"    : "sales_channels.voice_recording_completed",
    "품질점검결과"        : "quality_checks.check_result",
    "점검일"              : "quality_checks.check_date",
    "모집건수"            : "COUNT(sales_channels.channel_id)",
}

# YAML 저장
mapping_path = SL_DIR / "sql_mapping.yaml"
with open(mapping_path, "w", encoding="utf-8") as f:
    yaml.dump(SQL_MAPPING, f, allow_unicode=True, sort_keys=False)
print(f"SQL 매핑 저장: {mapping_path.name}")


# %% [5] Semantic Layer 컨텍스트 문자열 생성
def build_sl_context(sl_def: dict) -> str:
    """YAML 정의 → LLM에 전달할 텍스트 컨텍스트"""
    lines = ["[비즈니스 개념 정의]", ""]
    for cube in sl_def["cubes"]:
        lines.append(f"## {cube['name']}")
        lines.append(f"설명: {cube['description']}")
        lines.append("")
        lines.append("### 차원(Dimensions)")
        for d in cube.get("dimensions", []):
            lines.append(f"  - {d['name']}: {d['description']}")
        lines.append("")
        lines.append("### 측정값(Measures)")
        for m in cube.get("measures", []):
            lines.append(f"  - {m['name']}: {m['description']}")
        lines.append("")
    return "\n".join(lines)

def build_mapping_context(mapping: dict) -> str:
    """SQL 매핑 → LLM에 전달할 참조 텍스트"""
    lines = ["[개념→SQL 매핑 참조표]",
             "※ 쿼리 생성 시 아래 매핑을 반드시 사용하세요.", ""]
    for concept, sql_expr in mapping.items():
        lines.append(f"  {concept} → {sql_expr}")
    return "\n".join(lines)

SL_CONTEXT      = build_sl_context(SEMANTIC_LAYER)
MAPPING_CONTEXT = build_mapping_context(SQL_MAPPING)

# 민감 컬럼 노출 여부 측정
SENSITIVE_TOKENS = [
    "diagnosis_code", "prior_diagnosis_code", "disability_type",
    "disability_subtype", "disability_body_part", "disability_payment_rate",
    "cdr_score", "primary_diagnosis", "death_cause", "icd_code",
    "total_paid_premium", "monthly_premium", "annual_premium",
    "surrender_value", "loan_balance", "paid_amount",
    "birth_date", "insured_name", "death_date", "missing_period_days",
    "fraud_type", "causation_proven", "violation_type",
]
exposed_in_sl = [t for t in SENSITIVE_TOKENS if t in SL_CONTEXT]
print(f"\n[Semantic Layer 조건] 개념 컨텍스트 노출 민감 원시 컬럼: {len(exposed_in_sl)}개")
if len(exposed_in_sl) == 0:
    print("  ✅ 민감 컬럼 원시 식별자 노출 0개 — Information Hiding 달성")
else:
    print(f"  ⚠️  노출된 컬럼: {exposed_in_sl}")

print(f"\n컨텍스트 크기: {len(SL_CONTEXT)}자 (개념정의) + {len(MAPPING_CONTEXT)}자 (매핑)")


# %% [6] 프롬프트 구성
SYSTEM_PROMPT_SL = """당신은 보험 데이터 분석 전문가입니다.
아래의 [비즈니스 개념 정의]와 [개념→SQL 매핑 참조표]를 사용하여
자연어 질문에 대한 정확한 SQLite 쿼리를 생성하세요.

규칙:
1. SELECT 쿼리만 생성하세요.
2. 반드시 [개념→SQL 매핑 참조표]의 매핑을 사용하여 실제 컬럼명으로 변환하세요.
3. SQL 쿼리만 출력하세요. 설명, 마크다운 코드블록(```), 주석 불필요.
4. 세미콜론(;)으로 끝내세요.
5. 날짜 함수는 DATE('now'), STRFTIME(), JULIANDAY()만 사용하세요.
6. SQRT, EXP, LOG 같은 수학 함수는 사용하지 마세요."""

def build_user_prompt_sl(question: str) -> str:
    return f"""{SL_CONTEXT}

{MAPPING_CONTEXT}

[질문]
{question}

위 질문에 답하는 SQLite 쿼리를 생성하세요. 매핑 참조표의 SQL 표현식을 그대로 사용하세요."""

# 프롬프트 저장
prompt_sl_path = ROOT / "prompts" / "prompt_semantic_layer.txt"
prompt_sl_path.write_text(
    f"[SYSTEM]\n{SYSTEM_PROMPT_SL}\n\n[USER TEMPLATE]\n{build_user_prompt_sl('[QUESTION]')}",
    encoding="utf-8"
)
print(f"\n프롬프트 저장: {prompt_sl_path.name}")


# %% [7] 실행 정확도 함수 (02와 동일)
def normalize_result(rows: list) -> list:
    normalized = []
    for row in rows:
        norm_row = []
        for val in row:
            if isinstance(val, float):
                norm_row.append(round(val, 4))
            else:
                norm_row.append(val)
        normalized.append(tuple(norm_row))
    return sorted(normalized)

def execution_accuracy(pred_sql: str, gold_sql: str, conn: sqlite3.Connection) -> dict:
    result = {"ex": 0, "pred_rows": 0, "gold_rows": 0, "exec_error": None}
    try:
        gold_rows = normalize_result(conn.execute(gold_sql).fetchall())
        result["gold_rows"] = len(gold_rows)
    except Exception as e:
        result["exec_error"] = f"gold_sql 오류: {e}"
        return result
    try:
        pred_rows = normalize_result(conn.execute(pred_sql).fetchall())
        result["pred_rows"] = len(pred_rows)
        result["ex"] = 1 if pred_rows == gold_rows else 0
    except Exception as e:
        result["exec_error"] = f"pred_sql 오류: {e}"
    return result


# %% [8] LLM 호출 함수
def call_llm_sl(question: str) -> dict:
    t0 = time.perf_counter()
    try:
        resp = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT_SL},
                {"role": "user",   "content": build_user_prompt_sl(question)},
            ],
            temperature=0,
            timeout=TIMEOUT_S,
        )
        raw = resp.choices[0].message.content.strip()
        latency = (time.perf_counter() - t0) * 1000
        sql = raw
        for tag in ["```sql", "```sqlite", "```"]:
            sql = sql.replace(tag, "")
        sql = sql.strip().rstrip(";") + ";"
        return {"sql": sql, "latency_ms": round(latency, 2), "error": None, "raw": raw}
    except Exception as e:
        latency = (time.perf_counter() - t0) * 1000
        return {"sql": "", "latency_ms": round(latency, 2), "error": str(e), "raw": ""}


# %% [9] 실험 실행
print("\n" + "=" * 55)
print(f"  Semantic Layer 실험 시작")
print(f"  질문: {len(all_qs)}개 × {N_REPEAT}회 = {len(all_qs)*N_REPEAT}회 API 호출")
print("=" * 55)

conn    = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")
records = []
total   = len(all_qs) * N_REPEAT

with tqdm(total=total, desc="Semantic Layer", unit="call") as pbar:
    for q in all_qs:
        qid    = q["question_id"]
        q_text = q["question_ko"]
        gold   = q.get("gold_sql", "")
        cat    = q.get("category", "")
        diff   = q.get("difficulty", "")
        sil    = q.get("sensitive_info_level", "")

        for rep in range(1, N_REPEAT + 1):
            res = call_llm_sl(q_text)

            if res["error"] or not res["sql"]:
                ex_result = {"ex": 0, "pred_rows": 0, "gold_rows": 0,
                             "exec_error": res["error"]}
            else:
                ex_result = execution_accuracy(res["sql"], gold, conn)

            records.append({
                "question_id"         : qid,
                "category"            : cat,
                "difficulty"          : diff,
                "sensitive_info_level": sil,
                "iteration"           : rep,
                "method"              : "semantic_layer",
                "pred_sql"            : res["sql"],
                "gold_sql"            : gold,
                "ex"                  : ex_result["ex"],
                "pred_rows"           : ex_result["pred_rows"],
                "gold_rows"           : ex_result["gold_rows"],
                "latency_ms"          : res["latency_ms"],
                "api_error"           : res["error"],
                "exec_error"          : ex_result.get("exec_error"),
            })

            pbar.update(1)
            time.sleep(DELAY_SEC)

conn.close()
df_sl = pd.DataFrame(records)
print(f"\n실험 완료: {len(df_sl)}건")


# %% [10] 중간 결과 저장
raw_path = RAW_DIR / "semantic_layer_results_raw.csv"
df_sl.to_csv(raw_path, index=False, encoding="utf-8-sig")
print(f"원시 결과 저장: {raw_path.name}")


# %% [11] 정확도 분석
print("\n" + "=" * 55)
print("  Semantic Layer 실험 결과")
print("=" * 55)

overall_ex = df_sl["ex"].mean()
print(f"\n  전체 Execution Accuracy (EX): {overall_ex:.3f} ({overall_ex*100:.1f}%)")
print(f"  총 API 오류                 : {df_sl['api_error'].notna().sum()}건")
print(f"  총 실행 오류                : {df_sl['exec_error'].notna().sum()}건")
print(f"  평균 응답시간               : {df_sl['latency_ms'].mean():.1f} ms")
print(f"  중간 응답시간               : {df_sl['latency_ms'].median():.1f} ms")

diff_order = ["simple", "moderate", "challenging"]
print("\n  [난이도별 EX]")
diff_ex = (
    df_sl.groupby("difficulty")["ex"]
    .agg(["mean","count","sum"])
    .reindex(diff_order)
    .rename(columns={"mean":"EX","count":"호출수","sum":"정답수"})
)
diff_ex["EX(%)"] = (diff_ex["EX"] * 100).round(1)
print(diff_ex.to_string())

print("\n  [카테고리별 EX]")
cat_ex = (
    df_sl.groupby("category")["ex"]
    .agg(["mean","count","sum"])
    .sort_values("mean", ascending=False)
    .rename(columns={"mean":"EX","count":"호출수","sum":"정답수"})
)
cat_ex["EX(%)"] = (cat_ex["EX"] * 100).round(1)
print(cat_ex.to_string())

sil_order = ["high","medium","low"]
print("\n  [민감정보 수준별 EX]")
sil_ex = (
    df_sl.groupby("sensitive_info_level")["ex"]
    .agg(["mean","count"])
    .reindex(sil_order)
    .rename(columns={"mean":"EX","count":"호출수"})
)
sil_ex["EX(%)"] = (sil_ex["EX"] * 100).round(1)
print(sil_ex.to_string())


# %% [12] 반복별 안정성
print("\n  [반복별 EX — 안정성]")
iter_ex = df_sl.groupby("iteration")["ex"].mean()
for i, v in iter_ex.items():
    print(f"    Iteration {i}: EX = {v:.3f} ({v*100:.1f}%)")
std_iter = iter_ex.std()
print(f"    반복 간 표준편차: σ = {std_iter:.4f}")


# %% [13] 오류 상세
exec_err_df = df_sl[df_sl["exec_error"].notna()][
    ["question_id","difficulty","exec_error"]
].drop_duplicates("question_id")
if len(exec_err_df) > 0:
    print(f"\n  [실행 오류 — {len(exec_err_df)}개 질문]")
    print(exec_err_df.to_string(index=False))
else:
    print("\n  ✅ 실행 오류 없음")


# %% [14] 민감정보 노출 비교 (핵심: RQ2)
print("\n" + "=" * 55)
print("  민감정보 노출 비교 — RQ2 핵심 결과")
print("=" * 55)

# Text-to-SQL 결과 로드
t2s_summary_path = RAW_DIR / "text2sql_summary.json"
if t2s_summary_path.exists():
    with open(t2s_summary_path, encoding="utf-8") as f:
        t2s_summary = json.load(f)
    t2s_exposed = t2s_summary.get("sensitive_cols_exposed", 23)
else:
    t2s_exposed = 23

sl_exposed = len(exposed_in_sl)
reduction  = ((t2s_exposed - sl_exposed) / t2s_exposed * 100) if t2s_exposed > 0 else 0

print(f"\n  {'방법':<20} {'노출 민감 컬럼':>12} {'감소율':>8}")
print(f"  {'─'*42}")
print(f"  {'Text-to-SQL':<20} {t2s_exposed:>12}개 {'—':>8}")
print(f"  {'Semantic Layer':<20} {sl_exposed:>12}개 {reduction:>7.1f}%")
print(f"\n  → 민감 컬럼 노출 {reduction:.0f}% 구조적 감소")
if sl_exposed == 0:
    print("  ✅ Information Hiding 원칙 완전 달성 (Parnas, 1972)")


# %% [15] 최종 요약 저장
summary_sl = {
    "method"                   : "semantic_layer",
    "model"                    : LLM_MODEL,
    "n_questions"              : len(all_qs),
    "n_repeat"                 : N_REPEAT,
    "overall_ex"               : round(overall_ex, 4),
    "overall_ex_pct"           : round(overall_ex * 100, 1),
    "avg_latency_ms"           : round(df_sl["latency_ms"].mean(), 1),
    "median_latency_ms"        : round(df_sl["latency_ms"].median(), 1),
    "iter_std"                 : round(std_iter, 4),
    "n_api_errors"             : int(df_sl["api_error"].notna().sum()),
    "n_exec_errors"            : int(df_sl["exec_error"].notna().sum()),
    "sensitive_cols_exposed"   : sl_exposed,
    "sensitive_cols_text2sql"  : t2s_exposed,
    "sensitive_col_reduction_pct": round(reduction, 1),
    "diff_ex"                  : diff_ex["EX(%)"].to_dict(),
    "cat_ex"                   : cat_ex["EX(%)"].to_dict(),
}

summary_path = RAW_DIR / "semantic_layer_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary_sl, f, ensure_ascii=False, indent=2)

print(f"\n요약 저장: {summary_path.name}")
print(f"원시 결과: {raw_path.name}")
print(f"\n다음: 04_results_analysis.ipynb 실행")

  03_experiment_semantic_layer
Model    : gpt-4o-mini
반복 횟수 : 3회

질문셋 로드: 80개
Semantic Layer 정의 저장: schema_context.yaml
SQL 매핑 저장: sql_mapping.yaml

[Semantic Layer 조건] 개념 컨텍스트 노출 민감 원시 컬럼: 0개
  ✅ 민감 컬럼 원시 식별자 노출 0개 — Information Hiding 달성

컨텍스트 크기: 3417자 (개념정의) + 3773자 (매핑)

프롬프트 저장: prompt_semantic_layer.txt

  Semantic Layer 실험 시작
  질문: 80개 × 3회 = 240회 API 호출


Semantic Layer: 100%|██████████████████████████████████████████████████████████████| 240/240 [08:23<00:00,  2.10s/call]


실험 완료: 240건
원시 결과 저장: semantic_layer_results_raw.csv

  Semantic Layer 실험 결과

  전체 Execution Accuracy (EX): 0.292 (29.2%)
  총 API 오류                 : 0건
  총 실행 오류                : 48건
  평균 응답시간               : 1797.5 ms
  중간 응답시간               : 1426.4 ms

  [난이도별 EX]
                   EX  호출수  정답수  EX(%)
difficulty                            
simple       0.357143   42   15   35.7
moderate     0.318841  138   44   31.9
challenging  0.183333   60   11   18.3

  [카테고리별 EX]
                EX  호출수  정답수  EX(%)
category                           
사기탐지      0.523810   21   11   52.4
고지의무      0.500000   18    9   50.0
장해리스크     0.358974   39   14   35.9
보험금지급     0.357143   42   15   35.7
계약심사      0.307692   39   12   30.8
계약관리      0.153846   39    6   15.4
보험료산출     0.100000   30    3   10.0
재무리스크     0.000000   12    0    0.0

  [민감정보 수준별 EX]
                            EX  호출수  EX(%)
sensitive_info_level                      
high                  0.594203   69   59.4
medium        